# Maverick coreference demo
This notebook shows how Maverick groups expressions that refer to the same entity, then demonstrates a simple replacement of later mentions with the first mention.

**Example:** “Barack Obama … The former president … He …”

In [1]:
from maverick import Maverick

model = Maverick(
    hf_name_or_path="sapienzanlp/maverick-mes-ontonotes",
    device="cpu"
)


sapienzanlp/maverick-mes-ontonotes loading


c:\Users\piotr\Documents\AStudia\SW\no-need-to-be-so-big\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\piotr\Documents\AStudia\SW\no-need-to-be-so-big\.venv\lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [4]:
text = (
    "Barack Obama traveled to Rome last week. "
    "The former president visited the Colosseum. "
    "He said the city reminded him of his first trip to Europe."
)

result = model.predict(text)

print("Original text:\n", text)
print("\nCoreference clusters:")
print("Coreference clusters:")

for cluster in result["clusters_token_text"]:
    print(" -> ".join(cluster))


Original text:
 Barack Obama traveled to Rome last week. The former president visited the Colosseum. He said the city reminded him of his first trip to Europe.

Coreference clusters:
Coreference clusters:
Barack Obama -> The former president -> He -> him -> his
Rome -> the city


In [5]:
def replace_later_mentions_with_first(result):
    tokens = result["tokens"].copy()
    clusters = result["clusters_token_offsets"]

    replacements = {}
    for cluster in clusters:
        if len(cluster) < 2:
            continue
        first_start, first_end = cluster[0]
        antecedent = " ".join(tokens[first_start:first_end + 1])
        for start, end in cluster[1:]:
            replacements[(start, end)] = antecedent

    # Work from right to left so earlier token positions remain valid.
    for (start, end), antecedent in sorted(
        replacements.items(), key=lambda item: item[0][0], reverse=True
    ):
        tokens[start:end + 1] = [antecedent]

    return " ".join(tokens)

resolved_text = replace_later_mentions_with_first(result)
print("Resolved text:\n", resolved_text)


Resolved text:
 Barack Obama traveled to Rome last week . Barack Obama visited the Colosseum . Barack Obama said Rome reminded Barack Obama of Barack Obama first trip to Europe .
